# Temporal Fusion Transformer for Multivariate Multi-Horizon Forecasting

Sources for reference:
- https://pytorch-forecasting.readthedocs.io/en/stable/tutorials/stallion.html#

- https://pytorch-forecasting.readthedocs.io/en/stable/api/pytorch_forecasting.models.temporal_fusion_transformer._tft.TemporalFusionTransformer.html#

In [1]:
# Collab installation
%pip install -q "pytorch-lightning>=1.9,<2.0" "pytorch-forecasting>=1.0" optuna torchmetrics scikit-learn pandas matplotlib seaborn plotly


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.5/829.5 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 391.5/391.5 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.0/846.0 kB 63.5 MB/s eta 0:00:00


In [2]:
import warnings

warnings.filterwarnings("ignore")

import os
import math
import pandas as pd
import numpy as np

import torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor
from lightning.pytorch.loggers import TensorBoardLogger

from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss, MAE, RMSE
from pytorch_forecasting.data import GroupNormalizer, MultiNormalizer


print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Device count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('GPU device:', torch.cuda.get_device_name(0))

Torch version: 2.9.0+cu126
CUDA available: True
Device count: 1
GPU device: Tesla T4


In [4]:
CSV_PATH = 'household_power_consumption_hourly_clean.csv'
raw_df = pd.read_csv(CSV_PATH)
raw_df.head()

,date,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,2006-12-16 17:00:00,4.223,0.229,234.644,18.100,0.0,0.528,16.861
1,2006-12-16 18:00:00,3.632,0.080,234.580,15.600,0.0,6.717,16.867
2,2006-12-16 19:00:00,3.400,0.085,233.233,14.503,0.0,1.433,16.683
3,2006-12-16 20:00:00,3.269,0.075,234.072,13.917,0.0,0.000,16.783
4,2006-12-16 21:00:00,3.056,0.077,237.159,13.047,0.0,0.417,17.217


## Feature engineering
TFT requires specific columns in dataset to learn.

In [5]:
# Feature engineering
fe_df = raw_df.copy()

# Parse datetime
fe_df['date'] = pd.to_datetime(fe_df['date'])
fe_df = fe_df.sort_values('date')

# Create continuous time index (required by PyTorch Forecasting)
fe_df['time_idx'] = (fe_df['date'] - fe_df['date'].min()).dt.total_seconds() // 3600
fe_df['time_idx'] = fe_df['time_idx'].astype(int)

# Calendar features
fe_df['hour'] = fe_df['date'].dt.hour.astype(str).astype("category")
fe_df['dayofweek'] = fe_df['date'].dt.dayofweek.astype(str).astype("category")
fe_df['month'] = fe_df['date'].dt.month.astype(str).astype("category")

# Single group id; if multiple groups add a column per entity
fe_df['group_id'] = "series_1"

fe_df.head()

,date,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,time_idx,hour,dayofweek,month,group_id
0,2006-12-16 17:00:00,4.223,0.229,234.644,18.100,0.0,0.528,16.861,0,17,5,12,series_1
1,2006-12-16 18:00:00,3.632,0.080,234.580,15.600,0.0,6.717,16.867,1,18,5,12,series_1
2,2006-12-16 19:00:00,3.400,0.085,233.233,14.503,0.0,1.433,16.683,2,19,5,12,series_1
3,2006-12-16 20:00:00,3.269,0.075,234.072,13.917,0.0,0.000,16.783,3,20,5,12,series_1
4,2006-12-16 21:00:00,3.056,0.077,237.159,13.047,0.0,0.417,17.217,4,21,5,12,series_1


## Experiment Configuration


In [6]:
# Experiment hyperparameters
SEQ_LEN = 336  # Lookback window (encoder length)
PRED_LENS = [96, 192, 336, 720]  # Prediction horizons to test
# PRED_LENS = [96]
BATCH_SIZE = 64
LEARNING_RATE = 0.03
MAX_EPOCHS = 30

# Data split ratios
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
# TEST_RATIO (implicit)

# Multitarget
TARGET_COLUMNS = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3"
]

print(f'Lookback: {SEQ_LEN} hours')
print(f'Prediction horizons: {PRED_LENS}')
print(f'Train/Val/Test split: {TRAIN_RATIO}/{VAL_RATIO}/{1-TRAIN_RATIO-VAL_RATIO}')

Lookback: 336 hours
Prediction horizons: [96, 192, 336, 720]
Train/Val/Test split: 0.7/0.15/0.15000000000000005


In [7]:
# Train/Val/Test split based on time_idx
n = len(fe_df)
train_end = int(n * TRAIN_RATIO)
val_end = int(n * (TRAIN_RATIO + VAL_RATIO))

train_df = fe_df.iloc[:train_end]
val_df = fe_df.iloc[train_end:val_end]
test_df = fe_df.iloc[val_end:]

print(f'Train: {len(train_df)} samples')
print(f'Val: {len(val_df)} samples')
print(f'Test: {len(test_df)} samples')
print(f'Total: {len(fe_df)} samples')

Train: 24212 samples
Val: 5188 samples
Test: 5189 samples
Total: 34589 samples


## Helper Functions for Dataset & Model Creation

In [8]:
def create_timeseries_dataset(data, max_encoder_length, max_prediction_length, time_idx_col='time_idx'):
    """
    Args:
        data: DataFrame with features
        max_encoder_length: Lookback window
        max_prediction_length: Forecast horizon
        time_idx_col: Name of time index column

    Returns:
        TimeSeriesDataSet
    """
    dataset = TimeSeriesDataSet(
        data,
        time_idx=time_idx_col,
        target=TARGET_COLUMNS,
        group_ids=['group_id'],
        min_encoder_length=max_encoder_length // 2,
        max_encoder_length=max_encoder_length,
        min_prediction_length=1,
        max_prediction_length=max_prediction_length,

        # Static categoricals (constant per group)
        static_categoricals=['group_id'],

        # Static reals (constant per group)
        static_reals=[],

        # Time-varying known categoricals (known in advance)
        time_varying_known_categoricals=['hour', 'dayofweek', 'month'],

        # Time-varying known reals (known in advance)
        time_varying_known_reals=['time_idx'],

        # Time-varying unknown categoricals (not known in advance)
        time_varying_unknown_categoricals=[],

        # Time-varying unknown reals (not known in advance) - these are the features
        time_varying_unknown_reals=TARGET_COLUMNS,

        target_normalizer = MultiNormalizer([
            GroupNormalizer(groups=["group_id"], transformation="softplus")
            for _ in TARGET_COLUMNS]),
        add_relative_time_idx=True,
        add_target_scales=True
    )
    return dataset

## Training Loop for Multiple Experiments

We'll iterate through each prediction length and train a separate TFT model.

In [9]:
# Storage for experiment results
experiment_results = {}
trained_models = {}

for pred_len in PRED_LENS:
    print(f'\n{"="*60}')
    print(f'EXPERIMENT: Lookback={SEQ_LEN}, Pred_len={pred_len}')
    print(f'{"="*60}\n')

    # Create datasets
    print('Creating TimeSeriesDataSet...')
    training = create_timeseries_dataset(
        train_df,
        max_encoder_length=SEQ_LEN,
        max_prediction_length=pred_len
    )

    validation = TimeSeriesDataSet.from_dataset(training, val_df, predict=True, stop_randomization=True)
    testing = TimeSeriesDataSet.from_dataset(training, test_df, predict=True, stop_randomization=True)

    # Create dataloaders
    train_dataloader = training.to_dataloader(train=True, batch_size=BATCH_SIZE, num_workers=0)
    val_dataloader = validation.to_dataloader(train=False, batch_size=BATCH_SIZE * 2, num_workers=0)
    test_dataloader = testing.to_dataloader(train=False, batch_size=BATCH_SIZE * 2, num_workers=0)

    print(f'Train batches: {len(train_dataloader)}')
    print(f'Val batches: {len(val_dataloader)}')
    print(f'Test batches: {len(test_dataloader)}')

    tft = TemporalFusionTransformer.from_dataset(
        training,
        learning_rate=LEARNING_RATE,
        hidden_size=32,
        dropout=0.1,
        hidden_continuous_size=16,
        loss=QuantileLoss(),
        log_interval=50,
    )

    print(f'Model parameters: {sum(p.numel() for p in tft.parameters()):,}')

    # Setup trainer
    model_name = f'TFT_seq{SEQ_LEN}_pred{pred_len}'
    logger = TensorBoardLogger('lightning_logs', name=model_name)

    early_stop_callback = EarlyStopping(
        monitor='val_loss',
        min_delta=1e-4,
        patience=4,
        verbose=False,
        mode='min'
    )

    lr_monitor = LearningRateMonitor(logging_interval='epoch')

    trainer = pl.Trainer(
        max_epochs=MAX_EPOCHS,
        accelerator='auto',
        devices=1,
        gradient_clip_val=0.1,
        callbacks=[lr_monitor, early_stop_callback],
        logger=logger,
        enable_progress_bar=True,
    )

    # Train
    print(f'\nTraining {model_name}...')
    trainer.fit(
        tft,
        train_dataloaders=train_dataloader,
        val_dataloaders=val_dataloader,
    )

    # Evaluate on test set
    print(f'\nEvaluating on test set...')
    test_metrics = trainer.test(tft, dataloaders=test_dataloader, verbose=False)

    # Store results
    experiment_results[pred_len] = {
        'test_metrics': test_metrics,
        'best_model_path': trainer.checkpoint_callback.best_model_path,
        'model_name': model_name
    }
    trained_models[pred_len] = tft

    print(f'\nCompleted pred_len={pred_len}')
    print(f'Test loss: {test_metrics[0]["test_loss"]:.4f}')
    print(f'Best model: {trainer.checkpoint_callback.best_model_path}')

print('\n' + '='*60)
print('ALL EXPERIMENTS COMPLETED')
print('='*60)


EXPERIMENT: Lookback=336, Pred_len=96

Creating TimeSeriesDataSet...


INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores


Train batches: 379
Val batches: 1
Test batches: 1
Model parameters: 110,981

Training TFT_seq336_pred96...


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ MultiLoss                       │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │    324 │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │    768 │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │ 32.2 K │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │ 19.6 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │  4.5 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │  8.4 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │  8.4 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │  2.1 K │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │     64 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │  5.3 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │  2.6 K │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │  2.2 K │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │  2.2 K │ train │     0 │
│ 20 │ output_layer                       │ ModuleList                      │  1.6 K │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

...

Output()

INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()


Evaluating on test set...



Completed pred_len=720
Test loss: 7.8423
Best model: lightning_logs/TFT_seq336_pred720/version_0/checkpoints/epoch=6-step=2723.ckpt

ALL EXPERIMENTS COMPLETED


## Results Summary & Comparison

In [10]:
# Summarize results across all experiments
results_summary = []
for pred_len in PRED_LENS:
    res = experiment_results[pred_len]
    results_summary.append({
        'Prediction_Length': pred_len,
        'Test_Loss': res['test_metrics'][0]['test_loss'],
        'Model_Name': res['model_name'],
        'Best_Checkpoint': os.path.basename(res['best_model_path']) if res['best_model_path'] else 'N/A'
    })

results_df = pd.DataFrame(results_summary)
print('\n' + '='*70)
print('EXPERIMENT RESULTS SUMMARY')
print('='*70)
print(results_df.to_string(index=False))
print('='*70)


EXPERIMENT RESULTS SUMMARY
 Prediction_Length  Test_Loss         Model_Name        Best_Checkpoint
                96   5.147588  TFT_seq336_pred96 epoch=7-step=3032.ckpt
               192   6.433080 TFT_seq336_pred192 epoch=5-step=2286.ckpt
               336   6.517939 TFT_seq336_pred336 epoch=7-step=3064.ckpt
               720   7.842270 TFT_seq336_pred720 epoch=6-step=2723.ckpt


## Detailed Predictions & Visualizations

Select a specific prediction length to visualize predictions.

In [11]:
print('\n' + '='*70)
print('METRICS')
print('='*70)

for pred_len, tft in trained_models.items():

  predictions = tft.predict(
    test_dataloader, return_y=True, trainer_kwargs=dict(accelerator="cpu")
  )

  mae_array = np.array([
      MAE()(predictions.output[i], predictions.y[0][i])
      for i in range(len(TARGET_COLUMNS))
  ])

  mse_array = np.array([
      RMSE(reduction = "mean")(predictions.output[i], predictions.y[0][i]) # When reduction = "mean" it's a MSE.
      for i in range(len(TARGET_COLUMNS))
  ])

  print(f'Prediction Lenghth: {pred_len}')
  print(f'  MAE: {mae_array.mean()}')
  print(f'  MSE: {mse_array.mean()}')

print('='*70)

INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO: GPU available: True (cuda), used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores



METRICS


INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO: GPU available: True (cuda), used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores


Prediction Lenghth: 96
  MAE: 1.8678723573684692
  MSE: 16.172073364257812


INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO: GPU available: True (cuda), used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores


Prediction Lenghth: 192
  MAE: 1.8185007572174072
  MSE: 14.75873851776123


INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO: GPU available: True (cuda), used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores


Prediction Lenghth: 336
  MAE: 1.9235628843307495
  MSE: 17.439809799194336
Prediction Lenghth: 720
  MAE: 1.8047133684158325
  MSE: 14.931225776672363
